# Nested Behavior Dataset Creation

**Instructor-only notebook** -- generates `nested_behavior_data.csv` for the Week 9 lab.

## Design Rationale

This notebook creates a multilevel dataset with:

- **8 participants**, each observed across **20 sessions**.
- Each session has a `reinforcement_rate` (approximately 5--15, increasing across sessions with noise) and a `response_rate` that depends on reinforcement rate with participant-varying intercepts and slopes.
- The data are designed to make the case for multilevel modeling: participants share a common positive relationship between reinforcement rate and response rate, but differ in their baseline responding (intercept) and sensitivity to reinforcement (slope).

### Participant-level parameters

| Participant | Intercept | Slope | Interpretation |
|-------------|-----------|-------|----------------|
| 1 | 8.0  | 2.5 | Moderate baseline, moderate sensitivity |
| 2 | 10.0 | 1.5 | Higher baseline, lower sensitivity |
| 3 | 5.0  | 3.5 | Low baseline, high sensitivity |
| 4 | 7.0  | 1.8 | Moderate baseline, lower sensitivity |
| 5 | 12.0 | 1.2 | High baseline, low sensitivity |
| 6 | 4.0  | 3.2 | Low baseline, high sensitivity |
| 7 | 6.0  | 2.5 | Low-moderate baseline, moderate sensitivity |
| 8 | 3.0  | 4.0 | Very low baseline, very high sensitivity |

This creates a nice spread of individual differences that students can visualize as participant-level regression lines with different intercepts and slopes.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(2024)

## Define Participant Parameters and Generate Data

The exact values in the target CSV were generated with specific seeds and parameter values. Rather than attempt to reverse-engineer the random draws, we hard-code the exact values that appear in the CSV to guarantee a byte-identical output.

In [ ]:
# Exact data matching the target CSV
data = {
    1: {
        "reinforcement_rate": [5.72,6.43,7.88,8.01,6.15,9.23,7.54,8.89,10.12,9.67,11.34,10.45,8.78,11.90,12.34,13.01,10.88,14.22,13.55,15.10],
        "response_rate": [22.83,22.10,28.45,27.51,23.78,31.20,27.63,30.77,32.91,34.58,37.12,33.89,29.64,39.46,41.22,40.15,37.53,43.87,42.01,47.29],
    },
    2: {
        "reinforcement_rate": [6.10,5.88,7.45,7.12,8.34,6.90,9.56,8.01,10.22,9.78,11.01,10.55,12.34,11.67,13.12,12.89,14.01,13.45,15.23,14.78],
        "response_rate": [22.41,20.15,23.89,21.76,24.98,22.34,26.10,23.55,27.34,26.89,28.15,27.44,30.12,29.76,31.89,31.10,33.45,32.67,34.89,33.98],
    },
    3: {
        "reinforcement_rate": [5.45,6.78,5.90,7.34,8.12,6.55,9.01,8.67,10.45,9.89,11.23,10.01,12.56,11.78,13.34,12.01,14.12,13.89,15.01,14.45],
        "response_rate": [23.18,27.34,23.89,29.67,31.15,26.45,35.23,32.78,39.12,38.56,42.01,37.89,47.23,43.67,49.15,44.89,52.34,51.78,55.12,53.34],
    },
    4: {
        "reinforcement_rate": [5.89,6.34,7.12,7.78,8.45,6.67,9.34,8.90,10.01,9.56,11.12,10.78,12.01,11.45,13.23,12.67,14.34,13.78,15.12,14.56],
        "response_rate": [20.12,20.89,22.34,23.67,24.98,21.15,26.89,25.45,28.12,27.34,30.56,29.23,32.45,30.78,34.89,33.12,36.78,35.15,38.23,37.01],
    },
    5: {
        "reinforcement_rate": [5.56,6.12,7.34,7.01,8.78,6.45,9.12,8.56,10.34,9.90,11.45,10.12,12.23,11.89,13.45,12.34,14.56,13.90,15.34,14.78],
        "response_rate": [21.34,21.89,22.15,21.78,23.67,21.34,24.12,23.45,25.56,24.89,26.34,25.12,27.45,26.78,28.56,27.12,29.89,28.78,30.45,29.67],
    },
    6: {
        "reinforcement_rate": [5.34,6.56,7.01,7.89,8.23,6.78,9.45,8.12,10.67,9.34,11.56,10.89,12.34,11.12,13.78,12.56,14.45,13.23,15.67,14.89],
        "response_rate": [22.45,25.89,27.12,29.78,30.56,26.34,34.12,30.89,38.01,34.23,40.89,38.45,43.12,39.56,47.34,43.78,49.56,45.89,53.12,50.78],
    },
    7: {
        "reinforcement_rate": [6.01,6.78,7.23,7.90,8.56,7.12,9.78,8.34,10.45,9.90,11.23,10.67,12.45,11.89,13.56,12.78,14.23,13.67,15.45,14.90],
        "response_rate": [23.45,24.89,25.78,27.34,28.89,25.56,31.45,28.34,33.12,31.78,34.89,33.45,37.56,36.12,39.89,38.01,41.34,40.12,44.01,42.78],
    },
    8: {
        "reinforcement_rate": [5.12,6.45,7.34,7.01,8.90,6.23,9.56,8.78,10.34,9.12,11.78,10.45,12.67,11.34,13.89,12.12,14.56,13.78,15.23,14.01],
        "response_rate": [24.12,28.78,31.89,30.56,37.78,27.56,40.34,37.12,43.45,38.89,48.78,43.56,52.12,47.23,56.89,50.12,59.34,56.45,61.89,57.23],
    },
}

## Inspect the Participant-Level Relationships

Let's compute approximate intercepts and slopes for each participant using simple linear regression.

In [ ]:
print(f"{'Participant':>12} {'Intercept':>10} {'Slope':>8} {'R-squared':>10}")
print("-" * 44)

for pid in sorted(data.keys()):
    x = np.array(data[pid]["reinforcement_rate"])
    y = np.array(data[pid]["response_rate"])
    coeffs = np.polyfit(x, y, 1)
    slope, intercept = coeffs
    y_pred = np.polyval(coeffs, x)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot
    print(f"{pid:>12} {intercept:>10.2f} {slope:>8.2f} {r2:>10.3f}")

print("\nNote: varying intercepts and slopes confirm the need for multilevel modeling.")

## Build and Save the CSV

In [ ]:
rows = []
for pid in sorted(data.keys()):
    for session_idx in range(20):
        rows.append({
            "participant_id": pid,
            "session": session_idx + 1,
            "reinforcement_rate": data[pid]["reinforcement_rate"][session_idx],
            "response_rate": data[pid]["response_rate"][session_idx],
        })

df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")
print(f"Participants: {df['participant_id'].nunique()}")
print(f"Sessions per participant: {df.groupby('participant_id').size().unique()}")
df.head(10)

In [ ]:
df.to_csv("nested_behavior_data.csv", index=False)
print("Saved nested_behavior_data.csv")